# Week 45, Day 2: Neurons that accumulate: LIP and the random-dot task

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w045/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Build-up rate and coherence

Complete trials(c): on each 5 ms step after the latency, add k c dt plus noise to every undecided trial's x, stop trials whose |x| reaches 1, and record the rate 20 + 30 x (clipped at zero). Then measure the slope of the trial-averaged rate between 300 and 450 ms.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2001)
dt, lat, k = 0.005, 0.2, 8.0
T = int(1.0 / dt); time_ = np.arange(T) * dt

def trials(c, n=400):
    x = np.zeros(n); rate = np.zeros((n, T)); done = np.zeros(n, bool); rt = np.full(n, np.nan)
    for i in range(T):
        if time_[i] >= lat:
            idx = ~done
            ...
        rate[:, i] = np.clip(20 + 30 * x, 0, None)
    return rate, rt

cohs = [0.0, 0.128, 0.256, 0.512]
data, slope = {}, {}
i1, i2 = int(0.3 / dt), int(0.45 / dt)
for c in cohs:
    rate, rt = trials(c); data[c] = (rate, rt)
    m = rate.mean(0); slope[c] = (m[i2] - m[i1]) / 0.15
    plt.plot(time_, m, label=f"{100 * c:.1f}%")
print({f"{100 * c:.1f}%": round(s, 1) for c, s in slope.items()}, "spikes/s per second")
plt.xlabel("time from motion onset (s)"); plt.ylabel("rate (spikes/s)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_slopes():
    "the build-up slope increases with coherence and is near zero at 0%"
    s = [slope[c] for c in cohs]
    assert all(np.diff(s) > 0) and abs(s[0]) < 10

test_slopes()
print("All checks passed")

### Exercise: A common level at the decision

For each coherence, take the trials that reached the upper bound, find each one's rate at its decision time and average them; also record the mean decision time.

In [ ]:
final, mean_rt = {}, {}
for c in cohs:
    rate, rt = data[c]
    ...
    print(f"{100 * c:5.1f}%: rate at decision {final[c]:.1f} spikes/s, mean decision time {mean_rt[c]:.3f} s")

In [ ]:
import numpy as np
def test_common():
    "on trials ending at the upper bound, the rate at the decision is within 1 spike/s of 50 at every coherence"
    assert all(abs(final[c] - 50) < 1 for c in cohs)
def test_rt():
    "decisions are faster at higher coherence"
    assert mean_rt[0.512] < mean_rt[0.128]

test_common()
test_rt()
print("All checks passed")

### Exercise: Same mean, different variance

Write step_trials(c): each trial's rate is 20 until a step time (the latency plus an exponential delay with rate 2 + 20 c per second), then 50 with probability 0.5 + 0.75 c and 5 otherwise. Compare means and across-trial variances with the ramping model at 25.6% coherence.

In [ ]:
def step_trials(c, n=400):
    rate = np.full((n, T), 20.0)
    ts = lat + r.exponential(1 / (2 + 20 * c), n)
    up = r.random(n) < 0.5 + 0.75 * c
    for j in range(n):
        ...
    return rate

step_rate = step_trials(0.256); ramp_rate = data[0.256][0]
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].plot(time_, ramp_rate.mean(0), label="ramp"); ax[0].plot(time_, step_rate.mean(0), label="step"); ax[0].set_title("trial average"); ax[0].legend()
ax[1].plot(time_, ramp_rate.var(0), label="ramp"); ax[1].plot(time_, step_rate.var(0), label="step"); ax[1].set_title("variance across trials")
for a_ in ax: a_.set_xlabel("time (s)")
plt.show()
i = int(0.5 / dt)
print(f"at 0.5 s: mean ramp {ramp_rate[:, i].mean():.1f}, step {step_rate[:, i].mean():.1f}; variance ramp {ramp_rate[:, i].var():.0f}, step {step_rate[:, i].var():.0f}")

In [ ]:
import numpy as np
def test_mean():
    "both models' averages rise smoothly over the trial"
    for m in [step_rate.mean(0), ramp_rate.mean(0)]:
        assert m[-1] - m[int(0.3 / dt)] > 3
def test_var():
    "mid-trial, the step model's across-trial variance is at least three times the ramp model's"
    i = int(0.5 / dt)
    assert step_rate[:, i].var() > 3 * ramp_rate[:, i].var()

test_mean()
test_var()
print("All checks passed")